# AI worker Colab T4 batch

Batch runner cho đánh giá chất lượng, benchmark sampling và xuất result bundle. Colab không phải worker online: notebook không kết nối PostgreSQL, MinIO hay Milvus của máy local. Kết quả là file JSON và bundle có checksum, tải về rồi import local bằng `person-search-storage import-bundle`.

Chuẩn bị trên Google Drive: repository, checkpoint RaSa đặt tại `backend/config/model_artifacts/`, dataset WILDTRACK và `bundle_jobs.json` (camera/area/job/config ID lấy từ PostgreSQL local). Chọn runtime GPU T4 trước khi chạy.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_ROOT = Path('/content/drive/MyDrive/capstone_project_new')
DATASET_ROOT = Path('/content/drive/MyDrive/wildtrack-dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/ai-worker-batch')
BACKEND = PROJECT_ROOT / 'backend'
REGISTRY = BACKEND / 'config' / 'models.example.json'
ARTIFACT_ROOT = BACKEND / 'config'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert (BACKEND / 'pyproject.toml').is_file(), 'Mount Drive and place the repository first.'
os.environ.update({'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2', 'TOKENIZERS_PARALLELISM': 'false'})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cài đúng bộ dependency đã khóa bằng hash cho Linux x86_64/Python 3.12, sau đó cài backend không kéo thêm dependency. Nếu Colab đổi phiên bản Python, sinh lại lock bằng `tools/lock_colab_requirements.py --python-version <x.y>` và commit trước khi chạy.

In [ ]:
assert sys.version_info[:2] == (3, 12), f'Lock targets cp312, runtime is {sys.version}'
lock = BACKEND / 'requirements' / 'colab-linux-x86_64-cp312.lock.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--require-hashes', '-r', str(lock)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(BACKEND)], check=True)

In [ ]:
preflight = OUTPUT_DIR / 'colab-t4-preflight.json'
completed = subprocess.run([
    sys.executable, str(BACKEND / 'tools' / 'ai_preflight.py'),
    '--registry', str(REGISTRY),
    '--resource-config', str(BACKEND / 'config' / 'ai_resources.json'),
    '--profile', 'colab_t4', '--workspace', '/content',
    '--output', str(preflight),
], cwd=BACKEND, text=True, capture_output=True)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr

Đánh giá chất lượng trên WILDTRACK `Image_subsets` (AIW-26). Kết quả ghi lineage đầy đủ; tóm tắt số đo vào plan sau khi review.

In [ ]:
subprocess.run([
    sys.executable, str(BACKEND / 'tools' / 'evaluate_wildtrack.py'),
    '--dataset-root', str(DATASET_ROOT),
    '--queries', str(PROJECT_ROOT / 'files' / 'wildtrack_evaluation_queries.json'),
    '--registry', str(REGISTRY), '--artifact-root', str(ARTIFACT_ROOT),
    '--config-root', str(BACKEND / 'config'),
    '--output', str(OUTPUT_DIR / 'wildtrack-evaluation-colab-t4.json'),
], cwd=BACKEND, check=True)

Benchmark `N=10`/`N=20` trên T4 (AIW-27). Giới hạn 18000 frame nguồn (khoảng 5 phút video 60 fps) để vừa một phiên Colab; tăng nếu còn quota.

In [ ]:
subprocess.run([
    sys.executable, str(BACKEND / 'tools' / 'benchmark_sampling.py'),
    '--registry', str(REGISTRY), '--artifact-root', str(ARTIFACT_ROOT),
    '--config-root', str(BACKEND / 'config'),
    '--video', str(DATASET_ROOT / 'cam1.mp4'),
    '--intervals', '10,20', '--repeats', '3', '--max-source-frames', '18000',
    '--device', 'cuda', '--profile', 'colab_t4',
    '--output', str(OUTPUT_DIR / 'benchmark-colab-t4.json'),
], cwd=BACKEND, check=True)

Xuất result bundle cho từng video. `bundle_jobs.json` là danh sách object gồm `video`, `camera_id`, `area_id`, `job_id`, `config_id`, `timeline_origin`, `sampling_interval`. Mỗi video chạy riêng nên runtime bị ngắt chỉ cần chạy lại video còn thiếu; bundle đã có sidecar hợp lệ được bỏ qua.

In [ ]:
jobs = json.loads((OUTPUT_DIR / 'bundle_jobs.json').read_text())
for job in jobs:
    bundle = OUTPUT_DIR / 'bundles' / f"{Path(job['video']).stem}.json"
    if bundle.with_suffix('.lineage.json').is_file():
        print('skip', bundle.name)
        continue
    subprocess.run([
        sys.executable, str(BACKEND / 'tools' / 'export_result_bundle.py'),
        '--registry', str(REGISTRY), '--artifact-root', str(ARTIFACT_ROOT),
        '--config-root', str(BACKEND / 'config'),
        '--video', str(DATASET_ROOT / job['video']),
        '--camera-id', job['camera_id'], '--area-id', job['area_id'],
        '--job-id', job['job_id'], '--config-id', job['config_id'],
        '--timeline-origin', job['timeline_origin'],
        '--sampling-interval', str(job['sampling_interval']),
        '--device', 'cuda', '--output', str(bundle),
    ], cwd=BACKEND, check=True)

In [ ]:
import hashlib
manifest = {}
for path in sorted((OUTPUT_DIR / 'bundles').glob('*.json')):
    if path.name.endswith('.lineage.json'):
        continue
    manifest[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
(OUTPUT_DIR / 'bundles' / 'SHA256SUMS.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))